# Comparing Full and Fisher `logp` values

In [1]:
derive_errors = False  # 12/8/2025

if derive_errors:
    from cocoa_fisher import np, FisherMeta, FisherCase

    for space in ["fourier", "real"]:
        meta_As = FisherMeta(space, to_sigma8=False)
        base_As = FisherCase(meta_As)
        base_As.compute_values_and_errors(store_mats=False)

        print(space, meta_As.get_sigma8("As_1e9", meta_As.params["As_1e9"]))
        # fourier 0.8253647303125673
        # real 0.7752506017181334

        meta_σ8 = FisherMeta(space, to_sigma8=True)
        base_σ8 = FisherCase(meta_σ8)
        base_σ8.compute_values_and_errors(store_mats=True)

        np.savez(f"dc1_{space}_v1_err.npz",
                 errors_As=base_As.errors_MAP,
                 fisher_σ8=base_σ8.fisher+base_σ8.priors)

In [2]:
compute_logps = False  # 12/8/2025
compare_logps = False  # 12/9/2025

if compute_logps or compare_logps:
    from cocoa_dvcalc import np, camb, SPACE, DVCalc, Params
    from IPython.display import clear_output; clear_output(wait=False)
    with open(f"./projects/cocoa_dvcalc/dc1_{SPACE}_v1_err.npz", "rb") as f:
        vars().update(**dict(np.load(f)))  # errors_As, fisher_σ8

    # 12/9/2025: Convert h0 back to H0.
    errors_As[2] *= 100.0
    fisher_σ8[2, :] /= 100.0
    fisher_σ8[:, 2] /= 100.0

    def get_sigma8(self):  # self is a Provider instance.
        pars = camb.set_params(**self.camb_cache)  # Not recommended!
        pars.set_dark_energy(w=self.get_param("w"), wa=self.get_param("wa"), dark_energy_model="ppf")
        pars.set_matter_power(redshifts=[0.0], kmax=self.camb_cache["kmax"], silent=True)
        return camb.get_results(pars).get_sigma8_0()

In [3]:
if compute_logps:
    import pandas as pd
    from tqdm import tqdm

    params = Params(DVCalc(), f"./projects/cocoa_dvcalc/dc1_{SPACE}_v1_map.json")
    param_vec = np.array(list(params.params.values()))
    print(SPACE, params.logp, sigma8_0 := get_sigma8(params.provider))
    rng = np.random.default_rng(seed=42)

    data = np.zeros((NR := 1000, 4))  # NR: number of realizations
    for i in tqdm(range(NR)):
        delta_As = errors_As*rng.uniform(low=-3, high=3, size=31)
        data[i, 0] = params.try_values(Params.COSMOLOGY, (param_vec+delta_As)[:5])
        data[i, 1] = params.try_values(Params.ALLPARAMS,  param_vec+delta_As     )
        delta_σ8 = np.concatenate([[get_sigma8(params.provider) - sigma8_0], delta_As[1:]])
        data[i, 2] = -0.5 * np.einsum("i,ij,j->", delta_σ8[:5], fisher_σ8[:5, :5], delta_σ8[:5])
        data[i, 3] = -0.5 * np.einsum("i,ij,j->", delta_σ8    , fisher_σ8        , delta_σ8    )

    df = pd.DataFrame(data, columns=["logp_cosmo_full", "logp_all_full",
                                    "logp_cosmo_fisher", "logp_all_fisher"])
    df.to_csv(f"./projects/cocoa_dvcalc/dc1_{SPACE}_v1_logp.csv", index=False)
    clear_output(wait=False)

In [4]:
if compare_logps:
    params = Params(DVCalc(), f"./projects/cocoa_dvcalc/dc1_{SPACE}_v1_map.json")
    param_vec = np.array(list(params.params.values()))
    print(SPACE, logp_0 := params.logp, sigma8_0 := get_sigma8(params.provider))
    print()

    for i, param in enumerate(params.ALLPARAMS):
        print(param, params.try_value(param, param_vec[i] + errors_As[i]), "(+1σ)", end=" ")
        delta_σ8 = np.zeros_like(param_vec)
        delta_σ8[0] = get_sigma8(params.provider) - sigma8_0
        if param != "As_1e9": delta_σ8[i] = errors_As[i]
        print(-0.5 * np.einsum("i,ij,j->", delta_σ8, fisher_σ8, delta_σ8) + logp_0, "(Fisher)", end=" ")
        print(params.try_value(param, param_vec[i] - errors_As[i]), "(-1σ)")

In [5]:
if compare_logps:
    %matplotlib inline
    %config InlineBackend.figure_format = "retina"
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    def explore_param(param: str, RES: int = 17, ax: mpl.axes._axes.Axes = None):
        idx = params.ALLPARAMS.index(param)
        logp_full = np.empty(RES)
        logp_fisher = np.empty(RES)

        scales = np.linspace(-2, 2, RES)
        for i, scale in enumerate(scales):
            logp_full[i] = params.try_value(param, param_vec[idx] + errors_As[idx]*scale)
            delta_σ8 = np.zeros_like(param_vec)
            delta_σ8[0] = get_sigma8(params.provider) - sigma8_0
            if param != "As_1e9": delta_σ8[idx] = errors_As[idx]*scale
            logp_fisher[i] = -0.5 * np.einsum("i,ij,j->", delta_σ8, fisher_σ8, delta_σ8) + logp_0

        if ax is not None:
            ax.plot(errors_As[idx]*scales, logp_full, label="Full")
            ax.plot(errors_As[idx]*scales, logp_fisher, label="Fisher")
            ax.set_title(param)

In [6]:
if compare_logps:
    fig, axs = plt.subplots(2, 3, figsize=(10.8, 6.0))

    for i, param in enumerate(params.COSMOLOGY + ["roman_A1_2"]):
        explore_param(param, RES=17, ax=axs.flat[i])
    for j in range(2): axs[j, 0].set_ylabel("Log Probability")
    for i in range(3): axs[1, i].set_xlabel("Delta Parameter")

    axs[0, 0].legend()
    fig.tight_layout()
    plt.show()

In [7]:
visualize_logps = False  # 12/9/2025

if visualize_logps:
    %matplotlib inline
    %config InlineBackend.figure_format = "retina"
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    import pandas as pd
    fourier = pd.read_csv(f"./dc1_fourier_v1_logp.csv")
    real = pd.read_csv(f"./dc1_real_v1_logp.csv")

    import numpy as np
    rng = np.random.default_rng(seed=42)
    scales = np.zeros((NR := 1000, 31))  # NR: number of realizations
    for i in range(NR):
        scales[i] = rng.uniform(low=-3, high=3, size=31)

In [8]:
if False:  # 12/9/2025
    fig, axs = plt.subplots(2, 2, figsize=(9.0, 7.8), sharex="col")

    for i, df in enumerate([fourier, real]):
        logp_min = min(df.min())
        for j, middle in enumerate(["cosmo", "all"]):
            ax = axs[j, i]; ax.set_aspect("equal", adjustable="box")
            ax.scatter(df[f"logp_{middle}_full"], df[f"logp_{middle}_fisher"], c=scales[:, 2], s=1)
            ax.plot([logp_min, 0], [logp_min, 0], color="r", linestyle="--", zorder=5)

            ax.set_title(["Fourier", "Real"][i] + ": " + ["Cosmology only", "All parameters"][j])
            if j == 1: ax.set_xlabel("Full $\ln {\cal P}$")
            if i == 0: ax.set_ylabel("Fisher $\ln {\cal P}$")
            ax.set_xlim(-10000, 0); ax.set_ylim(-10000, 0)

    plt.tight_layout()
    plt.show()

In [9]:
if visualize_logps:  # 12/19/2025
    fig, axs = plt.subplots(2, 2, figsize=(8.4, 6.0), sharex="col", sharey="col")

    for j, df in enumerate([fourier, real]):
        for i, zoom in enumerate(["in", "out"]):
            logp_min = -10000 if zoom == "out" else -2000

            ax = axs[j, i]; ax.set_aspect("equal", adjustable="box")
            ax.scatter(df[f"logp_all_full"], df[f"logp_all_fisher"], c=scales[:, 2], s=1)
            ax.plot([logp_min, 0], [logp_min, 0], color="r", linestyle="--", zorder=5)

            ax.set_title(["Fourier", "Real"][j] + f": Zoomed-{zoom}")
            if j == 1:
                if i == 0: ax.set_xlabel("Full $\ln {\cal P}$")
                elif i == 1: ax.set_xlabel("Full $\ln {\cal P}$", labelpad=-2.5)
            if i == 0: ax.set_ylabel("Fisher $\ln {\cal P}$")
            if zoom == "out": ax.plot([-2000, -2000, 0], [0, -2000, -2000], c="k", ls=":")
            ax.set_xlim(logp_min, 0); ax.set_ylim(logp_min, 0)

            ax.minorticks_on()
            ax.xaxis.set_ticks_position('both')
            ax.yaxis.set_ticks_position('both')
            ax.patch.set_alpha(0.0)

    norm = mpl.colors.Normalize(vmin=-3, vmax=3)
    fig.colorbar(mpl.cm.ScalarMappable(norm=norm), ax=axs, aspect=40, pad=0.035,
                 label=r"$\Delta H_0 / {\rm marginalized}\ \sigma (H_0)$")
    axs[1, 1].tick_params(axis="x", labelrotation=15, pad=0)

    # plt.tight_layout()
    # plt.show()
    fig.savefig(f"fisher_plots/logp_check.pdf", bbox_inches="tight")
    plt.close(fig)

# Tabulating Comoving Wavenumbers at Angular Scales

In [10]:
tabulate_scales = False  # 12/8/2025, 5/5/2026

if tabulate_scales:
    import numpy as np
    from astropy import units as u
    from cocoa_dvcalc import SPACE, Provider, DVCalc, Params
    print(Provider.INPUT_FILE); Params.POSTERIOR = False

    from IPython.display import clear_output; clear_output(wait=False)
    params = Params(DVCalc(), f"./projects/cocoa_dvcalc/dc1_{SPACE}_v1.json")
    print(params.logp)  # This runs Provider.get_camb_results.

In [11]:
if tabulate_scales:
    # from cocoa_fisher import FisherViz
    # real = FisherViz(space="real", to_sigma8=False)
    # fourier = FisherViz(space="fourier", to_sigma8=False)

    if SPACE == "real":
        theta = np.array([
            2.97200132245545, 4.0400089913792465, 5.491812041638084, 7.465329796304372, 10.148043768622234,
            13.794808151791631, 18.75206062208114, 25.490733448767543, 34.65099142176315, 47.1030466394507,
            64.02982747918657, 87.03935519068173, 118.31750373639919, 160.835654856868, 218.63297531081636])
        ell = np.pi / u.arcmin.to(u.rad, theta)

    elif SPACE == "fourier":
        ell = np.array([
            35.31445806308525, 48.93449698153415, 67.80749659411795, 93.95941264291237, 130.19756911313132,
            180.4120154240586, 249.99311070922303, 346.4101615137752, 480.0132278028123, 665.1441685740268,
            921.6761942439148, 1277.147191799222, 1769.7158282998685, 2452.257760926468, 3398.0416685322934])
        theta = u.rad.to(u.arcmin, np.pi / ell)

    def format_scale(scale):
        if scale < 1e1: return f"{scale:.3f}"
        elif scale < 1e2: return f"{scale:.2f}"
        elif scale < 1e3: return f"{scale:.1f}"
        elif scale < 1e4: return f"{scale:.0f}"

    end = lambda scale: " & " if scale < 14 else " \\\\\n"

In [12]:
if tabulate_scales:
    dndz = np.loadtxt(f"./projects/roman_cpip_data_challenge/data_challenge1_{SPACE}_medium/challenge1.nz").T

    def weighted_mean_and_std(values, weights):
        average = np.average(values, weights=weights)
        variance = np.average(np.square(values - average), weights=weights)
        return average, np.sqrt(variance)

    means, stds = np.zeros(8), np.zeros(8)
    for tomo in range(1, 9):
        means[tomo-1], stds[tomo-1] = weighted_mean_and_std(dndz[0], weights=dndz[tomo])

    chi = params.dvcalc.provider.results.angular_diameter_distance(means) * (1+means)
    # comoving angular diameter distances in Mpc
    Rc_arr = np.outer(chi, 1 / ell) * params.params["H0"] / 100  # comoving wavelengths in h^-1 Mpc
    kc_arr = 1 / (Rc_arr * u.Mpc.to(u.Gpc))  # comoving wavenumbers in h Gpc^-1

In [13]:
if tabulate_scales:  # 12/8/2025, 5/5/2026
    print(r"        Bin", end=" & ")
    for scale in range(15): print(f"${scale+1}$", end=end(scale))
    print(r"    \hline")

    # 5/5/2026
    if SPACE == "fourier":
        print(r"        $\ell_{\max}$", end=" & ")
        for scale, ellmax in enumerate(np.geomspace(30, 4000, 16)[1:]):
            print(f"${format_scale(ellmax)}$", end=end(scale))
    elif SPACE == "real":
        print(r"        $\theta_{\min} \,[{\rm arcmin}]$", end=" & ")
        for scale, thetamin in enumerate(np.geomspace(2.5, 250, 16)[:-1]):
            print(f"${format_scale(thetamin)}$", end=end(scale))

    print(r"        $\bar{\ell}$", end=" & ")
    for scale in range(15): print(f"${format_scale(ell[scale])}$", end=end(scale))
    print(r"        $\bar{\theta} \,[{\rm arcmin}]$", end=" & ")
    for scale in range(15): print(f"${format_scale(theta[scale])}$", end=end(scale))
    print(r"    \hline")

    for tomo in range(1, 9):
        print(rf"        $\bar{{z}} = {means[tomo-1]:.3f}$", end=" & ")
        for scale in range(15):
            # print(f"${kc_arr[tomo-1, scale]:.1e}$".replace("e+0", r"{\rm e}"), end=end(scale))

            # 5/5/2026
            if SPACE == "fourier":
                print(f"${format_scale(kc_arr[tomo-1, scale])}$", end=end(scale))
            elif SPACE == "real":
                print(f"${format_scale(Rc_arr[tomo-1, scale])}$", end=end(scale))